<a href="https://colab.research.google.com/github/aminrafi394/ai-second-project/blob/Q2_Task2/Section2_Part2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Preproccesing

# Model Artichecture

**در این بخش قصد داریم از معماری DitilBert استفاده کنیم**

**اولین قدم در این مسیر این هست که ابتدا واژه های  مشابه با واژه های گارانتی را پیدا کنیم در این صورت می توانیم از دیتاست های و word emdeing های از پیش آموزش داده توسط سایر مدل ها  استفاده کنیم  
حال ما از دیتاست glove-wiki-gigaword-300  استفاده می کنیم**

**ابتدا دیتا ست های مربوط به تشابه واژه ها را دانلود می کنیم**

In [6]:
!pip install gensim

In [7]:
import gensim.downloader as api

similarity_api = api.load("glove-wiki-gigaword-300")

**در این بخش  ایتدا واژه های مشابه با ریشه های اولیه را پیدا می کنیم. در این بخش حداقل میزان امتیاز لازم ببرای این موضوع ۰.۶ در نظر گرفته شده ( این مقدار قابل مقدار دهی هست)  
در مرحله بعد به سراغ جدا سازی داده ها می رویم به این صورت که اگر داده های موحود تعداد دو یا بیشتر از این واژه ها داشته باشند جداسازی می شوند.**

In [10]:
startting_seed: list = ["warranty", "guarantee"]

def extract_similar_words(seed_words, similarity_api, min_point):
  founded_words: dict = {}
  for seed in startting_seed:
    if seed not in similarity_api:
      continue
    for word, similarity_point in similarity_api.most_similar(seed):
      if similarity_point >= min_point:
        founded_words[word] = similarity_point

    # sorting the words based on the scores ascendingly
    founded_words = dict(sorted(founded_words.items(), key=lambda item: item[1], reverse=True))
  return founded_words

similar_words: dict = extract_similar_words(startting_seed, similarity_api, 0.6)
for word in similar_words:
  print(f"{word} : {similar_words[word]}")


guarantees : 0.8646911382675171
guaranteed : 0.821403980255127
warranties : 0.753125011920929
guaranteeing : 0.7505971789360046
ensure : 0.6916533708572388
assure : 0.6704778075218201
insure : 0.6495670676231384
ensuring : 0.6334716081619263
secure : 0.6004204154014587


**در این بخش در نظر داریم تا یک سری لایه را اضافه کنیم تا در صورت مشاهده غلط های ملایی و تلفی واژه از دست نرود  
مثلا اگر یک حرف به اشتباه حذف شد از تعداد واژه های موثر کاشته نشود**

In [14]:
!pip install rapidfuzz

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 13.8 MB/s  0:00:00


In [15]:
from rapidfuzz import fuzz, process

def in_efective_word(similar_words: dict, input_word: str, treshold: int):
  if input_word in similar_words:
    return True
  for word in similar_words:
    if fuzz.ratio(word, input_word) >= treshold:
      return True
  return False


**در این بخش فلط های آوایی رو حدا می کنیم و در نظر می گیریم به این صورت که اگر دو واژه کد های اوایی یکسانی و یا کمتر از بیشترین مقدار فاصله را داشته باشند jelifish تولید می شوند را داشته باشند به عنوان واژه موثر شناخته می شوند در این صورت از این فیلتر برای برداشت آنها استفاده می شود در خالی این واژه ها توسط rapidfuzz رد می شدند و در نظر گرفته نمی شدند**

In [16]:
!pip install jellyfish

In [18]:
import jellyfish

phonetic_similar_words: dict = {word : jellyfish.soundex(word) for word in similar_words}

def phonetic_match(similar_words: dict, word: str, max_distance):
    word = jellyfish.metaphone(word)

    # the leventian distance has been used here.
    for key, phonetic_word in similar_words.items():
        distance = jellyfish.levenshtein_distance(word, phonetic_word)
        if distance <= max_distance:
            return True
    return False
